# least popular heros in the marvel graph

In [ ]:
from pyspark.sql import SparkSession, functions as F 
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder.appName('Superheros').getOrCreate()

# making the schema
schema = StructType([
    StructField('id', IntegerType()),
    StructField('name', StringType())
])

# reading in both the files needed
names = spark.read.schema(schema).option('sep', ' ').csv('../../MarvelNames.txt')
lines = spark.read.text('../../MarvelGraph.txt')


# grabbing the lines dataframe and returning each id with number of connections
connections = lines.withColumn('id', F.split(F.col('value'), ' ')[0]) \
.withColumn('connections', F.size(F.split(F.col('value'), ' ')) - 1) \
.groupBy('id').agg(F.sum('connections').alias('connections')) # groups all the id and counts total number of connections

# get the connections number for the least popular hero
leastPopAmount = connections.agg(F.min(F.col('connections'))).first()[0]

# get all the ids with that connection that is the least
leastPopularTable = connections.filter(F.col('connections') == leastPopAmount)

# join the leastPopularTable with the name table in order to get the name of each id
nameJoin = leastPopularTable.join(names, on='id', how='inner')

# only select the name and the connections for the table being shown and sort alphabetically
nameJoin.select('name', 'connections').sort('name', ascending = True).show()

+--------------------+-----------+
|                name|connections|
+--------------------+-----------+
|        BERSERKER II|          1|
|              BLARE/|          1|
|     CALLAHAN, DANNY|          1|
|       CLUMSY FOULUP|          1|
|         DEATHCHARGE|          1|
|              FENRIS|          1|
|GERVASE, LADY ALYSSA|          1|
|      GIURESCU, RADU|          1|
|JOHNSON, LYNDON BAIN|          1|
|                KULL|          1|
|          LUNATIK II|          1|
|MARVEL BOY II/MARTIN|          1|
|MARVEL BOY/MARTIN BU|          1|
|              RANDAK|          1|
|         RED WOLF II|          1|
|                RUNE|          1|
|         SEA LEOPARD|          1|
|           SHARKSKIN|          1|
|              ZANTOR|          1|
+--------------------+-----------+



In [7]:
from pyspark.sql import SparkSession, functions as func 
from pyspark.sql.types import StructField, StructType, StringType, IntegerType, DoubleType


spark = SparkSession.builder.appName('test').getOrCreate()

schema = StructType([
    StructField("customerId", IntegerType()),
    StructField('orderId', IntegerType()),
    StructField('orderTotal', DoubleType())
])

# reading in the data with the given schema
df = spark.read.csv('../../customer-orders.csv', schema=schema, header=False)


# getting the total amount per customer
total = df.groupBy(func.col('customerId')).agg(func.sum('orderTotal').alias('total_spent'))
df.show()

total.orderBy(func.col('total_spent'), ascending=False).show()

+----------+-------+----------+
|customerId|orderId|orderTotal|
+----------+-------+----------+
|        44|   8602|     37.19|
|        35|   5368|     65.89|
|         2|   3391|     40.64|
|        47|   6694|     14.98|
|        29|    680|     13.08|
|        91|   8900|     24.59|
|        70|   3959|     68.68|
|        85|   1733|     28.53|
|        53|   9900|     83.55|
|        14|   1505|      4.32|
|        51|   3378|      19.8|
|        42|   6926|     57.77|
|         2|   4424|     55.77|
|        79|   9291|     33.17|
|        50|   3901|     23.57|
|        20|   6633|      6.49|
|        15|   6148|     65.53|
|        44|   8331|     99.19|
|         5|   3505|     64.18|
|        48|   5539|     32.42|
+----------+-------+----------+
only showing top 20 rows
+----------+------------------+
|customerId|       total_spent|
+----------+------------------+
|        68| 6375.449999999997|
|        73| 6206.199999999999|
|        39| 6193.109999999999|
|        54| 60

In [ ]:
from pyspark import SparkContext, SparkConf
import prettytable

conf = SparkConf().setAppName("test")
sc = SparkContext.getOrCreate(conf)

rdd = sc.textFile("../../customer-orders.csv")

rdd2 = rdd.map(lambda x: float(x.split(',')[2])+10)

for value in rdd2.collect():
    print(value)


47.19
75.89
50.64
24.98
23.08
34.59
78.68
38.53
93.55
14.32
29.8
67.77000000000001
65.77000000000001
43.17
33.57
16.490000000000002
75.53
109.19
74.18
42.42
35.66
14.16
34.129999999999995
98.64
67.91
82.62
66.06
38.010000000000005
107.22
90.7
81.9
36.47
39.760000000000005
32.980000000000004
82.95
37.06
96.56
30.31
94.57
74.42
87.77
50.78
70.53999999999999
13.92
40.71
40.760000000000005
11.17
26.96
66.68
52.3
34.07
71.0
69.97999999999999
103.3
70.62
12.86
51.99
34.66
22.990000000000002
44.38
19.35
50.34
55.77
66.05
10.82
95.49
101.75
19.990000000000002
62.91
97.17
40.65
63.92
29.87
27.76
20.4
78.68
61.12
70.37
46.87
19.68
76.76
33.629999999999995
16.48
58.23
90.6
28.59
39.05
29.82
36.25
81.24
74.16
28.28
79.51
81.99
38.629999999999995
81.52
72.75999999999999
61.49
61.88
63.14
49.22
94.57
61.04
83.5
33.42
28.23
63.53
33.42
57.44
43.14
100.22
35.94
45.96
71.15
64.77000000000001
37.06
63.13
89.84
84.86
74.92
92.78
97.23
21.4
101.92
33.58
49.41
49.54
38.620000000000005
102.23
60.72
21.75999